In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Pairplot da variável alvo com as features numéricas
# Carregamos o dataframe original sem encoding para ter as colunas numéricas originais e a target
df = pd.read_csv('censo_limpo.csv')
num_features = ['idade', 'anos_estudo', 'ganho_capital', 'perda_capital', 'horas_por_semana', 'target']

plt.figure(figsize=(10, 10))
sns.pairplot(df[num_features], hue='target', palette='Set1', diag_kind='kde')
plt.savefig('pairplot_alvo_numericas.png')
plt.close()
print("Pairplot gerado com sucesso.")

Pairplot gerado com sucesso.


<Figure size 1000x1000 with 0 Axes>

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, cohen_kappa_score, roc_auc_score, roc_curve, confusion_matrix

df = pd.read_csv('censo_limpo.csv')

# Drop peso_final as decided previously or check if needed
if 'peso_final' in df.columns:
    df = df.drop(columns=['peso_final'])
if 'renda' in df.columns:
    df = df.drop(columns=['renda'])

print("Colunas antes do get_dummies:", df.columns.tolist())

# Categorical columns get_dummies
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Colunas categóricas:", cat_cols)

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print("Colunas após get_dummies:", df_encoded.shape)

X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# Numerical columns for scaling
# Which are numeric?
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Colunas numéricas para padronização:", num_cols)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Best scaling: StandardScaler (standardization)
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Treino/Teste preparados com sucesso!")

Colunas antes do get_dummies: ['idade', 'classe_trabalho', 'escolaridade', 'anos_estudo', 'estado_civil', 'ocupacao', 'relacionamento', 'raca', 'sexo', 'ganho_capital', 'perda_capital', 'horas_por_semana', 'pais_natal', 'target']
Colunas categóricas: ['classe_trabalho', 'escolaridade', 'estado_civil', 'ocupacao', 'relacionamento', 'raca', 'sexo', 'pais_natal']
Colunas após get_dummies: (30139, 96)
Colunas numéricas para padronização: ['idade', 'anos_estudo', 'ganho_capital', 'perda_capital', 'horas_por_semana']
Treino/Teste preparados com sucesso!


In [21]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier

# ==============================================================================
# MODELO 1: RANDOM FOREST
# ==============================================================================
print('=== 1. RANDOM FOREST ===')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predições Treino e Teste
y_pred_tr = rf_model.predict(X_train)
y_pred_te = rf_model.predict(X_test)
y_prob_tr = rf_model.predict_proba(X_train)[:, 1]
y_prob_te = rf_model.predict_proba(X_test)[:, 1]

# Métricas Treino
acc_tr = accuracy_score(y_train, y_pred_tr)
prec_tr = precision_score(y_train, y_pred_tr)
rec_tr = recall_score(y_train, y_pred_tr)
auc_tr = roc_auc_score(y_train, y_prob_tr)

# Métricas Teste
acc_te = accuracy_score(y_test, y_pred_te)
prec_te = precision_score(y_test, y_pred_te)
rec_te = recall_score(y_test, y_pred_te)
auc_te = roc_auc_score(y_test, y_prob_te)

print(
    f'Acurácia -> Treino: {acc_tr:.4f} | Teste: {acc_te:.4f} | Dif (Treino -'
    f' Teste): {acc_tr - acc_te:.4f}'
)
print(
    f'ROC AUC  -> Treino: {auc_tr:.4f} | Teste: {auc_te:.4f} | Dif (Treino -'
    f' Teste): {auc_tr - auc_te:.4f}'
)
print(
    f'Precisão -> Treino: {prec_tr:.4f} | Teste: {prec_te:.4f} | Dif (Treino -'
    f' Teste): {prec_tr - prec_te:.4f}'
)
print(
    f'Recall   -> Treino: {rec_tr:.4f} | Teste: {rec_te:.4f} | Dif (Treino -'
    f' Teste): {rec_tr - rec_te:.4f}'
)
print('-' * 60)

=== 1. RANDOM FOREST ===
Acurácia -> Treino: 0.9771 | Teste: 0.8464 | Dif (Treino - Teste): 0.1307
ROC AUC  -> Treino: 0.9974 | Teste: 0.8932 | Dif (Treino - Teste): 0.1042
Precisão -> Treino: 0.9638 | Teste: 0.7167 | Dif (Treino - Teste): 0.2471
Recall   -> Treino: 0.9434 | Teste: 0.6336 | Dif (Treino - Teste): 0.3098
------------------------------------------------------------


In [22]:
# ==============================================================================
# MODELO 2: BAGGING COM REGRESSÃO LOGÍSTICA
# ==============================================================================
print('=== 2. BAGGING COM REGRESSÃO LOGÍSTICA ===')
lr_base = LogisticRegression(max_iter=1000, random_state=42)
bagging_lr_model = BaggingClassifier(
    estimator=lr_base, n_estimators=100, random_state=42
)
bagging_lr_model.fit(X_train, y_train)

y_pred_tr = bagging_lr_model.predict(X_train)
y_pred_te = bagging_lr_model.predict(X_test)
y_prob_tr = bagging_lr_model.predict_proba(X_train)[:, 1]
y_prob_te = bagging_lr_model.predict_proba(X_test)[:, 1]

acc_tr = accuracy_score(y_train, y_pred_tr)
prec_tr = precision_score(y_train, y_pred_tr)
rec_tr = recall_score(y_train, y_pred_tr)
auc_tr = roc_auc_score(y_train, y_prob_tr)

acc_te = accuracy_score(y_test, y_pred_te)
prec_te = precision_score(y_test, y_pred_te)
rec_te = recall_score(y_test, y_pred_te)
auc_te = roc_auc_score(y_test, y_prob_te)

print(
    f'Acurácia -> Treino: {acc_tr:.4f} | Teste: {acc_te:.4f} | Dif (Treino -'
    f' Teste): {acc_tr - acc_te:.4f}'
)
print(
    f'ROC AUC  -> Treino: {auc_tr:.4f} | Teste: {auc_te:.4f} | Dif (Treino -'
    f' Teste): {auc_tr - auc_te:.4f}'
)
print(
    f'Precisão -> Treino: {prec_tr:.4f} | Teste: {prec_te:.4f} | Dif (Treino -'
    f' Teste): {prec_tr - prec_te:.4f}'
)
print(
    f'Recall   -> Treino: {rec_tr:.4f} | Teste: {rec_te:.4f} | Dif (Treino -'
    f' Teste): {rec_tr - rec_te:.4f}'
)
print('-' * 60)

=== 2. BAGGING COM REGRESSÃO LOGÍSTICA ===
Acurácia -> Treino: 0.8486 | Teste: 0.8461 | Dif (Treino - Teste): 0.0026
ROC AUC  -> Treino: 0.9077 | Teste: 0.9007 | Dif (Treino - Teste): 0.0070
Precisão -> Treino: 0.7386 | Teste: 0.7320 | Dif (Treino - Teste): 0.0066
Recall   -> Treino: 0.6070 | Teste: 0.6023 | Dif (Treino - Teste): 0.0047
------------------------------------------------------------


In [23]:
# ==============================================================================
# MODELO 3: ÁRVORE DE DECISÃO (Individual)
# ==============================================================================
print('=== 3. ÁRVORE DE DECISÃO (INDIVIDUAL) ===')
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_tr = dt_model.predict(X_train)
y_pred_te = dt_model.predict(X_test)
y_prob_tr = dt_model.predict_proba(X_train)[:, 1]
y_prob_te = dt_model.predict_proba(X_test)[:, 1]

acc_tr = accuracy_score(y_train, y_pred_tr)
prec_tr = precision_score(y_train, y_pred_tr)
rec_tr = recall_score(y_train, y_pred_tr)
auc_tr = roc_auc_score(y_train, y_prob_tr)

acc_te = accuracy_score(y_test, y_pred_te)
prec_te = precision_score(y_test, y_pred_te)
rec_te = recall_score(y_test, y_pred_te)
auc_te = roc_auc_score(y_test, y_prob_te)

print(
    f'Acurácia -> Treino: {acc_tr:.4f} | Teste: {acc_te:.4f} | Dif (Treino -'
    f' Teste): {acc_tr - acc_te:.4f}'
)
print(
    f'ROC AUC  -> Treino: {auc_tr:.4f} | Teste: {auc_te:.4f} | Dif (Treino -'
    f' Teste): {auc_tr - auc_te:.4f}'
)
print(
    f'Precisão -> Treino: {prec_tr:.4f} | Teste: {prec_te:.4f} | Dif (Treino -'
    f' Teste): {prec_tr - prec_te:.4f}'
)
print(
    f'Recall   -> Treino: {rec_tr:.4f} | Teste: {rec_te:.4f} | Dif (Treino -'
    f' Teste): {rec_tr - rec_te:.4f}'
)
print('-' * 60)

=== 3. ÁRVORE DE DECISÃO (INDIVIDUAL) ===
Acurácia -> Treino: 0.9771 | Teste: 0.8185 | Dif (Treino - Teste): 0.1586
ROC AUC  -> Treino: 0.9984 | Teste: 0.7742 | Dif (Treino - Teste): 0.2243
Precisão -> Treino: 0.9845 | Teste: 0.6436 | Dif (Treino - Teste): 0.3409
Recall   -> Treino: 0.9224 | Teste: 0.6076 | Dif (Treino - Teste): 0.3148
------------------------------------------------------------


In [24]:
# MODELO 4: REGRESSÃO LOGÍSTICA (Individual)
# ==============================================================================
print('=== 4. REGRESSÃO LOGÍSTICA (INDIVIDUAL) ===')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_tr = lr_model.predict(X_train)
y_pred_te = lr_model.predict(X_test)
y_prob_tr = lr_model.predict_proba(X_train)[:, 1]
y_prob_te = lr_model.predict_proba(X_test)[:, 1]

acc_tr = accuracy_score(y_train, y_pred_tr)
prec_tr = precision_score(y_train, y_pred_tr)
rec_tr = recall_score(y_train, y_pred_tr)
auc_tr = roc_auc_score(y_train, y_prob_tr)

acc_te = accuracy_score(y_test, y_pred_te)
prec_te = precision_score(y_test, y_pred_te)
rec_te = recall_score(y_test, y_pred_te)
auc_te = roc_auc_score(y_test, y_prob_te)

print(
    f'Acurácia -> Treino: {acc_tr:.4f} | Teste: {acc_te:.4f} | Dif (Treino -'
    f' Teste): {acc_tr - acc_te:.4f}'
)
print(
    f'ROC AUC  -> Treino: {auc_tr:.4f} | Teste: {auc_te:.4f} | Dif (Treino -'
    f' Teste): {auc_tr - auc_te:.4f}'
)
print(
    f'Precisão -> Treino: {prec_tr:.4f} | Teste: {prec_te:.4f} | Dif (Treino -'
    f' Teste): {prec_tr - prec_te:.4f}'
)
print(
    f'Recall   -> Treino: {rec_tr:.4f} | Teste: {rec_te:.4f} | Dif (Treino -'
    f' Teste): {rec_tr - rec_te:.4f}'
)
print('-' * 60)

=== 4. REGRESSÃO LOGÍSTICA (INDIVIDUAL) ===
Acurácia -> Treino: 0.8489 | Teste: 0.8461 | Dif (Treino - Teste): 0.0029
ROC AUC  -> Treino: 0.9076 | Teste: 0.9008 | Dif (Treino - Teste): 0.0068
Precisão -> Treino: 0.7384 | Teste: 0.7320 | Dif (Treino - Teste): 0.0065
Recall   -> Treino: 0.6093 | Teste: 0.6023 | Dif (Treino - Teste): 0.0071
------------------------------------------------------------


In [25]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier

# Dicionário contendo os 4 modelos solicitados
models = {
    'Random Forest (100 est.)': RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    'Bagging com Regressão Logística (100 est.)': BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=42),
        n_estimators=100,
        random_state=42,
    ),
    'Árvore de Decisão (Individual)': DecisionTreeClassifier(random_state=42),
    'Regressão Logística (Individual)': LogisticRegression(
        max_iter=1000, random_state=42
    ),
}

# Loop para treinar, prever e exibir a Matriz de Confusão de cada modelo
for name, model in models.items():
  print('=' * 80)
  print(f'MODELO: {name}')
  print('=' * 80)

  # 1. Treinamento
  model.fit(X_train, y_train)

  # 2. Predições para Treino e Teste
  y_pred_train = model.predict(X_train)
  y_pred_test = model.predict(X_test)

  # 3. Cálculo das Matrizes de Confusão
  cm_train = confusion_matrix(y_train, y_pred_train)
  cm_test = confusion_matrix(y_test, y_pred_test)

  # 4. Exibição didática
  print('Matriz de Confusão - TREINO:')
  print(
      '[[Verdadeiros Negativos (TN)  | Falsos Positivos (FP)]\n'
      ' [Falsos Negativos (FN)     | Verdadeiros Positivos (TP)]]'
  )
  print(cm_train)
  print('\nInterpretação Treino:')
  print(
      f' - Acertou que ganham <=50K (TN): {cm_train[0, 0]} | Errou (FP):'
      f' {cm_train[0, 1]}'
  )
  print(
      f' - Errou que ganham >50K (FN): {cm_train[1, 0]}  | Acertou (TP):'
      f' {cm_train[1, 1]}'
  )

  print('\n' + '-' * 80)

  print('Matriz de Confusão - TESTE:')
  print(
      '[[Verdadeiros Negativos (TN)  | Falsos Positivos (FP)]\n'
      ' [Falsos Negativos (FN)     | Verdadeiros Positivos (TP)]]'
  )
  print(cm_test)
  print('\nInterpretação Teste:')
  print(
      f' - Acertou que ganham <=50K (TN): {cm_test[0, 0]}  | Errou (FP):'
      f' {cm_test[0, 1]}'
  )
  print(
      f' - Errou que ganham >50K (FN): {cm_test[1, 0]}   | Acertou (TP):'
      f' {cm_test[1, 1]}'
  )
  print('\n\n')

MODELO: Random Forest (100 est.)
Matriz de Confusão - TREINO:
[[Verdadeiros Negativos (TN)  | Falsos Positivos (FP)]
 [Falsos Negativos (FN)     | Verdadeiros Positivos (TP)]]
[[17893   213]
 [  340  5665]]

Interpretação Treino:
 - Acertou que ganham <=50K (TN): 17893 | Errou (FP): 213
 - Errou que ganham >50K (FN): 340  | Acertou (TP): 5665

--------------------------------------------------------------------------------
Matriz de Confusão - TESTE:
[[Verdadeiros Negativos (TN)  | Falsos Positivos (FP)]
 [Falsos Negativos (FN)     | Verdadeiros Positivos (TP)]]
[[4151  376]
 [ 550  951]]

Interpretação Teste:
 - Acertou que ganham <=50K (TN): 4151  | Errou (FP): 376
 - Errou que ganham >50K (FN): 550   | Acertou (TP): 951



MODELO: Bagging com Regressão Logística (100 est.)
Matriz de Confusão - TREINO:
[[Verdadeiros Negativos (TN)  | Falsos Positivos (FP)]
 [Falsos Negativos (FN)     | Verdadeiros Positivos (TP)]]
[[16816  1290]
 [ 2360  3645]]

Interpretação Treino:
 - Acertou que g

Resumo dos Resultados Calculados (Treino vs. Teste):
Modelos Baseados em Regressão Logística (Bagging e Individual):

Apresentam diferenças extremamente baixas entre treino e teste (ex: Acurácia com diferença de apenas ~0.0028 e ROC AUC com ~0.0069), provando que são modelos lineares muito estáveis e sem overfitting.

Modelos Baseados em Árvores (Random Forest e Árvore de Decisão):

Apresentam uma diferença maior entre treino e teste (ex: Acurácia com diferença de ~0.13 a ~0.15 e ROC AUC com até ~0.22 na árvore individual), evidenciando o comportamento natural de árvores de decisão não podadas de memorizarem os dados de treino (overfitting estrutural).